In [1]:
import numpy as np
import get_klines
import treatment

In [2]:
px,vol= get_klines.find_existing_file('30m', '2017-01-01', '2026-10-10', 1000, "ret")  #[linhas[t,t-1],colunas] [t,stocks]

[1/776] ETHUSDT: loaded from cache
[2/776] BTCUSDT: loaded from cache
[3/776] USDCUSDT: loaded from cache
[4/776] SOLUSDT: loaded from cache
[5/776] BNBUSDT: loaded from cache
[6/776] ZECUSDT: loaded from cache
[7/776] PEPEUSDT: loaded from cache
[8/776] USD1USDT: loaded from cache
[9/776] NEARUSDT: loaded from cache
[10/776] XRPUSDT: loaded from cache
[11/776] TRXUSDT: loaded from cache
[12/776] HYPEUSDT: loaded from cache
[13/776] ENAUSDT: loaded from cache
[14/776] UNIUSDT: loaded from cache
[15/776] DOGEUSDT: loaded from cache
[16/776] SUIUSDT: loaded from cache
[17/776] LINKUSDT: loaded from cache
[18/776] UUSDT: loaded from cache
[19/776] WLDUSDT: loaded from cache
[20/776] FDUSDUSDT: loaded from cache
[21/776] SNDKBUSDT: loaded from cache
[22/776] INTCBUSDT: loaded from cache
[23/776] GOOGLBUSDT: loaded from cache
[24/776] EURUSDT: loaded from cache
[25/776] RLUSDUSDT: loaded from cache
[26/776] AAVEUSDT: loaded from cache
[27/776] XAUTUSDT: loaded from cache
[28/776] PARTIUSDT:

In [3]:
ret = px.pct_change(fill_method=None)

In [8]:
y=ret.values

In [75]:
def kalman_local_level(y, Q_hat, R_hat, x0=0.0, P0=np.nanvar(y)):
    x_prior = np.zeros((len(y), len(y[0,:])))
    x_update = np.zeros((len(y), len(y[0,:])))
    x_update[0,:] = x0
    P_prior = np.zeros((len(y), len(y[0,:])))
    P_update = np.zeros((len(y), len(y[0,:])))
    P_update[0,:] = P0
    e = np.zeros((len(y), len(y[0,:])))
    s = np.zeros((len(y), len(y[0,:])))
    K = np.zeros((len(y), len(y[0,:])))
    for t in range(1, len(y)):
        time_update_predict(t, x_prior, x_update, P_prior, P_update, Q_hat)
        measure_update(t, x_prior, x_update, y, P_prior, P_update, s, e, K, R_hat)
    return x_prior, x_update, P_prior, P_update, e, s

In [76]:
def time_update_predict(t,x_prior,x_update,P_prior,P_update,Q):
    x_prior[t,:] = x_update[t-1,:] #F*x_update[t-1,:]
    P_prior[t,:] = P_update[t-1,:] + Q #F*P_update[t-1,:]*np.linalg.matrix_transpose(F) + Q

def measure_update(t,x_prior,x_update, y, P_prior,P_update,s,e,K,R):
    e[t,:] = y[t,:] - x_prior[t,:]#H*x_prior[t,:]
    s[t,:] = P_prior[t,:]+R #H*P_prior[t,:]*np.linalg.matrix_transpose(H) + R
    K[t,:] = P_prior[t,:]/s[t,:] #*np.linalg.matrix_transpose(H)*np.linalg.inv(s[t,:])

    x_update[t,:] = x_prior[t,:] + K[t,:]*e[t,:]
    P_update[t,:] = (1 - K[t,:])*P_prior[t,:]

In [86]:
from scipy.optimize import minimize_scalar

R_i = np.var(y, axis=0)                      # per-coin, FIT WINDOW ONLY, then frozen

def neg_loglik_Q(logQ, y, R_i, burn=2000):
    Q = np.exp(logQ)                             # common Q (scalar), broadcasts over coins
    _, _, _, _, e, s = kalman_local_level(y, Q, R_i, x0=0.0, P0=R_i)
    e, s = e[burn:], s[burn:]
    ll = -0.5 * np.sum(np.log(2*np.pi) + np.log(s) + e**2 / s)
    return -ll / e.size

res = minimize_scalar(neg_loglik_Q, bounds=(-25, 5), args=(y, R_i), method="bounded")
Q_hat = np.exp(res.x)

In [4]:
import get_klines_fut

In [5]:
px_fut= get_klines_fut.find_existing_file('30m', '2017-01-01', '2026-10-10', 1000, "ret")  #[linhas[t,t-1],colunas] [t,stocks]

[1/522] HYPEUSDT: loaded from cache
[2/522] ETHUSDT: loaded from cache
[3/522] BTCUSDT: loaded from cache
[4/522] ZECUSDT: loaded from cache
[5/522] SOLUSDT: loaded from cache
[6/522] BNBUSDT: loaded from cache
[7/522] XRPUSDT: loaded from cache
[8/522] UNIUSDT: loaded from cache
[9/522] DOGEUSDT: loaded from cache
[10/522] NEARUSDT: loaded from cache
[11/522] ENAUSDT: loaded from cache
[12/522] TRXUSDT: loaded from cache
[13/522] SUIUSDT: loaded from cache
[14/522] PARTIUSDT: loaded from cache
[15/522] WLDUSDT: loaded from cache
[16/522] LINKUSDT: loaded from cache
[17/522] ADAUSDT: loaded from cache
[18/522] PUMPUSDT: loaded from cache
[19/522] AAVEUSDT: loaded from cache
[20/522] AVAXUSDT: loaded from cache
[21/522] 1000PEPEUSDT: loaded from cache
[22/522] LITUSDT: loaded from cache
[23/522] TAOUSDT: loaded from cache
[24/522] XMRUSDT: loaded from cache
[25/522] IDOLUSDT: loaded from cache
[26/522] ONDOUSDT: loaded from cache
[27/522] LSKUSDT: loaded from cache
[28/522] FILUSDT: loa

In [6]:
ret_fut=px_fut.pct_change(fill_method=None)

In [7]:
ret = px.pct_change(fill_method=None)

In [8]:
ret_fut

,BTCUSDT,ETHUSDT,SOLUSDT,ZECUSDT,HYPEUSDT,BNBUSDT,1000PEPEUSDT,XRPUSDT,NEARUSDT,ENAUSDT,...,ALCHUSDT,LQTYUSDT,JOEUSDT,BANANAUSDT,DOLOUSDT,BNTUSDT,WAXPUSDT,1000CATUSDT,MOCAUSDT,XVSUSDT
open_time,,,,,,,,,,,,,,,,,,,,,
2017-08-17 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-17 04:30:00+00:00,0.011118,0.007415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-17 05:00:00+00:00,0.002592,0.001989,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-17 05:30:00+00:00,-0.001083,0.002945,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-17 06:00:00+00:00,-0.000996,0.000033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-09 22:00:00+00:00,-0.000155,0.000571,0.000550,-0.005201,0.001547,-0.003004,-0.002655,0.000287,-0.000823,0.002799,...,0.001642,-0.001293,0.007772,0.002599,0.005407,-0.000621,0.000000,-0.001964,0.000522,-0.001291
2026-10-09 22:30:00+00:00,0.001356,0.000977,0.001100,0.001528,0.002020,0.002999,0.002206,0.001148,0.015657,0.000000,...,0.001171,0.002157,0.002571,0.001296,0.008605,0.004040,0.003448,0.003443,0.005008,0.002909
2026-10-09 23:00:00+00:00,-0.000193,0.000060,0.000275,0.000148,-0.000237,0.000700,0.000000,-0.000143,-0.002637,0.002326,...,0.002339,0.014636,0.000000,0.004660,0.004266,0.001857,0.006873,0.003922,0.004983,0.004834


In [9]:
ret

,BTCUSDT,USDCUSDT,ETHUSDT,SOLUSDT,BNBUSDT,ZECUSDT,PEPEUSDT,NEARUSDT,XRPUSDT,TRXUSDT,...,USDSOLDUSDT,BTTUSDT,USDSBUSDT,HCUSDT,BCHSVUSDT,ERDUSDT,VENUSDT,STORMUSDT,PAXUSDT,BCHABCUSDT
open_time,,,,,,,,,,,,,,,,,,,,,
2017-08-17 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-17 04:30:00+00:00,0.011118,NaN,0.007415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-17 05:00:00+00:00,0.002592,NaN,0.001989,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-17 05:30:00+00:00,-0.001083,NaN,0.002945,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-17 06:00:00+00:00,-0.000996,NaN,0.000033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-09 22:00:00+00:00,-0.000155,0.00000,0.000571,0.000550,-0.003004,-0.005201,-0.002525,-0.000823,0.000287,0.000602,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-09 22:30:00+00:00,0.001356,0.00002,0.000977,0.001100,0.002999,0.001528,0.002532,0.015657,0.001148,-0.000602,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-10-09 23:00:00+00:00,-0.000193,0.00001,0.000060,0.000275,0.000700,0.000148,0.000000,-0.002637,-0.000143,-0.000301,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
ret.std()

BTCUSDT       0.005498
USDCUSDT      0.000391
ETHUSDT       0.006823
SOLUSDT       0.009231
BNBUSDT       0.007499
                ...   
ERDUSDT       0.012374
VENUSDT       0.010200
STORMUSDT     0.013169
PAXUSDT       0.000774
BCHABCUSDT    0.009491
Length: 776, dtype: float64

In [16]:
h_ratio=ret.corrwith(ret_fut)*ret.std()/ret_fut.std()

In [24]:
h_ratio=h_ratio.dropna()